# Document Ingestion Pipeline — RAG Chunking Reference

**Purpose:** Reference implementation for converting technical PDF documents into high-quality semantic chunks ready for vector embedding and retrieval-augmented generation (RAG).

**Pipeline Overview:**

```
PDF Document
    │
    ▼
Docling Parser  ──►  Markdown
    │
    ▼
Section Extraction  (# / ## / ###)
    │
    ▼
Token Analysis  ──►  flag chunks > 100 tokens
    │
    ├── short sections  ──►  pass through as-is
    │
    └── long sections   ──►  semantic chunking
                                    │
                                    ▼
                            Final Chunks + Metadata
```

**Target Document:** `LIFTOFF_DOE_VPP_2023.pdf` (DOE Virtual Power Plant strategy report)

**Output:** `chunks` — a list of dicts ready for downstream embedding pipelines.

---
## 1. Environment Setup

Import all required libraries and define pipeline constants.

**Dependencies** (all pre-installed in the project venv):
- `docling` — PDF parsing and Markdown export
- `tiktoken` — token counting with cl100k_base encoding (used by OpenAI embedding models)
- `semchunk` — semantic-aware text chunking
- `nltk` — sentence boundary detection for fallback chunking

In [ ]:
import re
import json
from pathlib import Path
from typing import Any

import tiktoken
import semchunk
import nltk
import pandas as pd

from docling.document_converter import DocumentConverter

# Download NLTK sentence tokenizer data (only needed once)
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

# ── Pipeline Constants ────────────────────────────────────────────────────────

DATA_DIR = Path("data")                     # Directory containing source PDFs
TARGET_DOC = "Technology Strategy Assessment - Lithium-ion_0.pdf"    # Primary document for this notebook
ENCODING_NAME = "cl100k_base"               # tiktoken encoding (OpenAI embedding compatible)
TOKEN_THRESHOLD = 100                        # Max tokens before semantic chunking is applied
MIN_CHUNK_TOKENS = 20                        # Minimum tokens; smaller fragments are dropped

print("Environment ready.")
print(f"  Data directory : {DATA_DIR.resolve()}")
print(f"  Target document: {TARGET_DOC}")
print(f"  Token threshold: {TOKEN_THRESHOLD} tokens")

Environment ready.
  Data directory : /Users/atorrico/Documents/projects/chunking/data
  Target document: potential-benefits-high-power-high-capacity-batteries-january-2020.pdf
  Token threshold: 100 tokens


[nltk_data] Error loading punkt: <urlopen error [SSL:
[nltk_data]     CERTIFICATE_VERIFY_FAILED] certificate verify failed:
[nltk_data]     unable to get local issuer certificate (_ssl.c:1032)>
[nltk_data] Error loading punkt_tab: <urlopen error [SSL:
[nltk_data]     CERTIFICATE_VERIFY_FAILED] certificate verify failed:
[nltk_data]     unable to get local issuer certificate (_ssl.c:1032)>


---
## 2. Document Loading

`load_documents()` scans a directory for PDF files and returns their paths. Designed for extensibility — point it at any directory to process a new batch of documents.

For this notebook we target a single file, but the function supports processing multiple documents in a loop.

In [14]:
def load_documents(data_dir: Path) -> list[Path]:
    """
    Discover all PDF documents in the given directory.

    Args:
        data_dir: Path to the directory containing source PDF files.

    Returns:
        Sorted list of Path objects for each discovered PDF.
    """
    pdf_paths = sorted(data_dir.glob("*.pdf"))
    if not pdf_paths:
        raise FileNotFoundError(f"No PDF files found in '{data_dir.resolve()}'")
    return pdf_paths


# ── Load & Inspect ────────────────────────────────────────────────────────────

all_documents = load_documents(DATA_DIR)

print(f"Discovered {len(all_documents)} document(s):")
for doc in all_documents:
    size_mb = doc.stat().st_size / (1024 * 1024)
    print(f"  [{size_mb:5.1f} MB]  {doc.name}")

# Select the target document for this pipeline run
target_path = DATA_DIR / TARGET_DOC
assert target_path.exists(), f"Target document not found: {target_path}"

print(f"\nTarget: {target_path}")

Discovered 3 document(s):
  [ 10.9 MB]  LIFTOFF_DOE_VPP_2023.pdf
  [  1.0 MB]  Technology Strategy Assessment - Lithium-ion_0.pdf
  [  1.9 MB]  potential-benefits-high-power-high-capacity-batteries-january-2020.pdf

Target: data/potential-benefits-high-power-high-capacity-batteries-january-2020.pdf


---
## 3. Parsing with Docling

[Docling](https://github.com/DS4SD/docling) is an advanced document understanding library that preserves document structure during PDF parsing. It handles multi-column layouts, tables, headers, and figures — producing clean Markdown output suitable for downstream NLP tasks.

**Why Docling over PyPDF / pdfplumber?**
- Preserves heading hierarchy (`#`, `##`, `###`)
- Reconstructs table structure as Markdown tables
- Handles multi-column PDF layouts correctly
- Produces paragraph-coherent text (no mid-word line breaks)

> Note: First-time parsing may download Docling model weights. Subsequent runs use the local cache.

In [15]:
def parse_document_to_markdown(pdf_path: Path) -> str:
    """
    Parse a PDF document using Docling and export to Markdown.

    Docling preserves document structure including headers, lists, tables,
    and paragraph boundaries — critical for high-quality chunking downstream.

    Args:
        pdf_path: Path to the PDF file to parse.

    Returns:
        Markdown string representation of the document.
    """
    converter = DocumentConverter()
    result = converter.convert(str(pdf_path))
    markdown = result.document.export_to_markdown()
    return markdown


# ── Parse Document ────────────────────────────────────────────────────────────

print(f"Parsing '{target_path.name}' with Docling...")
markdown_text = parse_document_to_markdown(target_path)

print(f"\nParsing complete.")
print(f"  Total characters: {len(markdown_text):,}")
print(f"  Total lines     : {markdown_text.count(chr(10)):,}")
print("\n" + "─" * 60)
print("Markdown preview (first 2000 chars):")
print("─" * 60)
print(markdown_text[:2000])

Parsing 'potential-benefits-high-power-high-capacity-batteries-january-2020.pdf' with Docling...

Parsing complete.
  Total characters: 189,064
  Total lines     : 1,852

────────────────────────────────────────────────────────────
Markdown preview (first 2000 chars):
────────────────────────────────────────────────────────────
U.S. DEPARTMENT OF

<!-- image -->

## Potential Benefits of High-Power, High-Capacity Batteries

January 2020

United States Department of Energy Washington, DC 20585

## Executive Summary

The ability of the United States (U.S.) electric power system (i.e., the electric grid) to reliably meet customer demand is crucial to our economy and national security. The increasing adoption of variable generation technologies and dynamic changes in customer demand are creating the need for enhanced grid flexibility to ensure the continued reliability, resilience, and security of the electric power system. Batteries and other energy storage technologies that have the capa

---
## 4. Markdown Section Extraction

We exploit the hierarchical structure Docling preserves in the Markdown output. Headers (`#`, `##`, `###`) act as natural semantic boundaries — each header introduces a new topic, making them ideal chunk candidates.

**Strategy:**
1. Split the document on header lines using regex
2. Track the heading hierarchy to preserve parent context (e.g., a `###` subsection retains awareness of its parent `##` section)
3. Each extracted section becomes an initial candidate chunk

**Header context** is important for retrieval: a chunk titled only "Results" is ambiguous; "Executive Summary > Key Results" is not.

In [16]:
def extract_sections(markdown: str) -> list[dict[str, Any]]:
    """
    Split a Markdown document into sections based on heading hierarchy.

    Each section captures its heading text, heading level (1/2/3),
    body content, and the full ancestor heading path for retrieval context.

    Args:
        markdown: Full Markdown string from the document parser.

    Returns:
        List of section dicts with keys:
            - header   : The heading text (without # markers)
            - level    : Heading depth (1, 2, or 3)
            - content  : Body text under this heading
            - context  : Full breadcrumb path (e.g. "Intro > Background")
    """
    # Regex to match heading lines at levels 1–3
    heading_pattern = re.compile(r'^(#{1,3})\s+(.+)$', re.MULTILINE)

    # Split document into (heading_match, body_text) pairs
    sections: list[dict[str, Any]] = []
    matches = list(heading_pattern.finditer(markdown))

    # Handle leading content before the first heading
    if matches and matches[0].start() > 0:
        preamble = markdown[:matches[0].start()].strip()
        if preamble:
            sections.append({
                "header": "[Document Preamble]",
                "level": 0,
                "content": preamble,
                "context": "[Document Preamble]",
            })

    # Track ancestor headings for breadcrumb context
    ancestors: dict[int, str] = {}  # level -> heading text

    for i, match in enumerate(matches):
        level = len(match.group(1))      # Number of '#' characters
        header_text = match.group(2).strip()

        # Body text runs from end of this heading to start of next heading
        body_start = match.end()
        body_end = matches[i + 1].start() if i + 1 < len(matches) else len(markdown)
        body_text = markdown[body_start:body_end].strip()

        # Update ancestor tracking: clear any deeper levels
        ancestors[level] = header_text
        for deeper in [l for l in ancestors if l > level]:
            del ancestors[deeper]

        # Build breadcrumb context path
        breadcrumb = " > ".join(
            ancestors[l] for l in sorted(ancestors)
        )

        sections.append({
            "header": header_text,
            "level": level,
            "content": body_text,
            "context": breadcrumb,
        })

    return sections


# ── Extract Sections ──────────────────────────────────────────────────────────

sections = extract_sections(markdown_text)

print(f"Extracted {len(sections)} sections.")
print("\nFirst 10 sections:")
print(f"{'Level':<7} {'Header':<50} {'Content (chars)'}")
print("─" * 80)
for s in sections[:10]:
    lvl_str = "#" * s["level"] if s["level"] > 0 else "pre"
    print(f"{lvl_str:<7} {s['header'][:48]:<50} {len(s['content']):>6}")

Extracted 120 sections.

First 10 sections:
Level   Header                                             Content (chars)
────────────────────────────────────────────────────────────────────────────────
pre     [Document Preamble]                                    34
##      Potential Benefits of High-Power, High-Capacity        69
##      Executive Summary                                     646
##      Report Scope and Approach                           12856
##      POTENTIAL BENEFITS OF HIGH-POWER, HIGH-CAPACITY         0
##      Table of Contents                                    3878
##      I. Energy Storage Grid Functions                     5963
##      I.A. Bulk Energy Services                               0
##      Electric Energy Time shift (Arbitrage)                560
##      Electric Supply Capacity                              950


---
## 5. Markdown Chunking

With sections extracted, we now classify each section based on token count:

- **Short sections (≤ 100 tokens):** Pass through as a single chunk — already a good semantic unit.
- **Long sections (> 100 tokens):** Flag for semantic chunking in the next step.

We use `tiktoken` with the `cl100k_base` encoding — the same tokenizer used by OpenAI's `text-embedding-3-small` and `text-embedding-3-large` models. Token counts here directly correspond to what the embedding model will see.

> **Why 100 tokens?** Most high-performing RAG systems target chunk sizes of 100–512 tokens. 100 tokens (~75 words) is a safe lower bound that keeps chunks focused without losing context.

In [17]:
# ── Tokenizer Setup ───────────────────────────────────────────────────────────

encoding = tiktoken.get_encoding(ENCODING_NAME)


def count_tokens(text: str) -> int:
    """
    Count the number of tokens in a text string.

    Uses tiktoken cl100k_base — the encoding used by OpenAI embedding models
    (text-embedding-3-small, text-embedding-3-large, text-embedding-ada-002).

    Args:
        text: Input text string.

    Returns:
        Integer token count.
    """
    return len(encoding.encode(text))


def analyze_sections(
    sections: list[dict[str, Any]],
    threshold: int = TOKEN_THRESHOLD,
) -> list[dict[str, Any]]:
    """
    Annotate each section with token count and chunking flag.

    Adds two fields to each section dict:
        - token_count    : Number of tokens in the section content
        - needs_chunking : True if token_count exceeds threshold

    Args:
        sections : List of section dicts from extract_sections().
        threshold: Token count above which semantic chunking is applied.

    Returns:
        Annotated sections list (mutates in place, also returns for chaining).
    """
    for section in sections:
        tc = count_tokens(section["content"])
        section["token_count"] = tc
        section["needs_chunking"] = tc > threshold
    return sections


# ── Analyze Sections ──────────────────────────────────────────────────────────

sections = analyze_sections(sections)

token_counts = [s["token_count"] for s in sections]
needs_chunking = [s for s in sections if s["needs_chunking"]]
short_sections = [s for s in sections if not s["needs_chunking"]]

print("Token count distribution across sections:")
print(f"  Total sections         : {len(sections)}")
print(f"  Pass-through (≤ {TOKEN_THRESHOLD} tok) : {len(short_sections)}")
print(f"  Needs chunking (> {TOKEN_THRESHOLD} tok): {len(needs_chunking)}")
print(f"  Min tokens             : {min(token_counts)}")
print(f"  Max tokens             : {max(token_counts)}")
print(f"  Mean tokens            : {sum(token_counts) / len(token_counts):.1f}")

# Show a few sections that will be chunked
print(f"\nSample sections requiring chunking:")
for s in needs_chunking[:5]:
    print(f"  [{s['token_count']:>4} tok]  {s['header'][:60]}")

Token count distribution across sections:
  Total sections         : 120
  Pass-through (≤ 100 tok) : 47
  Needs chunking (> 100 tok): 73
  Min tokens             : 0
  Max tokens             : 2131
  Mean tokens            : 309.6

Sample sections requiring chunking:
  [ 109 tok]  Executive Summary
  [2131 tok]  Report Scope and Approach
  [ 225 tok]  Table of Contents
  [1011 tok]  I. Energy Storage Grid Functions
  [ 102 tok]  Electric Energy Time shift (Arbitrage)


---
## 6. Token Length Analysis — Distribution View

A DataFrame view of all sections to inspect the distribution before chunking. This helps validate whether the threshold choice is appropriate for this document.

In [18]:
# ── Section Summary Table ─────────────────────────────────────────────────────

df_sections = pd.DataFrame([
    {
        "level": s["level"],
        "header": s["header"][:60],
        "token_count": s["token_count"],
        "needs_chunking": s["needs_chunking"],
    }
    for s in sections
])

print("Section overview:")
print(df_sections.to_string(index=False))

print("\nToken count summary:")
print(df_sections["token_count"].describe().round(1).to_string())

Section overview:
 level                                                    header  token_count  needs_chunking
     0                                       [Document Preamble]           10           False
     2 Potential Benefits of High-Power, High-Capacity Batteries           16           False
     2                                         Executive Summary          109            True
     2                                 Report Scope and Approach         2131            True
     2 POTENTIAL BENEFITS OF HIGH-POWER, HIGH-CAPACITY BATTERIES            0           False
     2                                         Table of Contents          225            True
     2                          I. Energy Storage Grid Functions         1011            True
     2                                 I.A. Bulk Energy Services            0           False
     2                    Electric Energy Time shift (Arbitrage)          102            True
     2                                  El

---
## 7. Semantic Chunking

For sections exceeding the token threshold, we apply semantic chunking rather than naive character or token splitting.

**Primary strategy — `semchunk` library:**
`semchunk` splits text at natural semantic boundaries (sentences, paragraphs) while respecting a target token window. It guarantees:
- No mid-sentence splits
- Chunks stay within the token limit
- Prefers paragraph breaks over sentence breaks

**Fallback strategy — NLTK sentence grouping:**
If `semchunk` produces a fragment below the minimum token threshold (e.g., a very short sentence), we apply NLTK sentence tokenization and greedily group sentences until the window fills.

**Rules enforced:**
- Never split mid-sentence
- Drop fragments with fewer than `MIN_CHUNK_TOKENS` tokens
- Each sub-chunk inherits the parent section's header and context

In [19]:
def _split_sentences(text: str) -> list[str]:
    """
    Split text into sentences using regex — no NLTK data files required.

    Splits on sentence-ending punctuation (. ! ?) followed by whitespace
    and an uppercase letter, preserving the delimiter with the preceding sentence.

    Args:
        text: Input text to split.

    Returns:
        List of sentence strings.
    """
    # Split after . ! ? when followed by whitespace + capital or end-of-string
    parts = re.split(r'(?<=[.!?])\s+(?=[A-Z])', text)
    return [p.strip() for p in parts if p.strip()]


def _group_sentences_by_token_window(
    sentences: list[str],
    max_tokens: int,
    min_tokens: int,
) -> list[str]:
    """
    Greedily group sentences into windows that fit within max_tokens.

    Args:
        sentences : List of sentence strings.
        max_tokens: Maximum tokens per output group.
        min_tokens: Minimum tokens; groups below this are merged with the next.

    Returns:
        List of grouped text strings.
    """
    groups: list[str] = []
    current_sentences: list[str] = []
    current_tokens = 0

    for sentence in sentences:
        sentence_tokens = count_tokens(sentence)

        if current_tokens + sentence_tokens > max_tokens and current_sentences:
            groups.append(" ".join(current_sentences))
            current_sentences = [sentence]
            current_tokens = sentence_tokens
        else:
            current_sentences.append(sentence)
            current_tokens += sentence_tokens

    # Flush remaining sentences
    if current_sentences:
        remaining = " ".join(current_sentences)
        # If last group is too small, merge with previous group
        if groups and count_tokens(remaining) < min_tokens:
            groups[-1] = groups[-1] + " " + remaining
        else:
            groups.append(remaining)

    return groups


def semantic_chunk(
    section: dict[str, Any],
    max_tokens: int = TOKEN_THRESHOLD,
    min_tokens: int = MIN_CHUNK_TOKENS,
) -> list[dict[str, Any]]:
    """
    Split a long section into semantic sub-chunks.

    Primary strategy: semchunk library — splits at natural semantic boundaries
    (paragraph > sentence > word) while respecting the token limit.

    Fallback: regex sentence splitting + greedy token-window grouping.
    No external data files required.

    Args:
        section   : Section dict with keys: header, context, content.
        max_tokens: Target maximum tokens per output chunk.
        min_tokens: Minimum tokens; smaller fragments are dropped or merged.

    Returns:
        List of sub-chunk dicts inheriting the parent section's metadata.
    """
    content = section["content"]

    # ── Primary: semchunk ─────────────────────────────────────────────────────
    # chunkerify expects a token_counter: (str -> int), NOT a tokenizer (str -> list).
    # count_tokens returns an int — this is correct.
    try:
        chunker = semchunk.chunkerify(count_tokens, max_tokens)
        raw_chunks = chunker(content)
    except Exception as e:
        # Fallback: regex sentence splitting, no external dependencies
        sentences = _split_sentences(content)
        if not sentences:
            sentences = [content]
        raw_chunks = _group_sentences_by_token_window(sentences, max_tokens, min_tokens)

    # ── Filter: drop or merge sub-minimum fragments ───────────────────────────
    filtered_chunks: list[str] = []
    for chunk in raw_chunks:
        chunk = chunk.strip()
        if not chunk:
            continue
        tc = count_tokens(chunk)
        if tc < min_tokens:
            if filtered_chunks:
                filtered_chunks[-1] = filtered_chunks[-1] + " " + chunk
            continue
        filtered_chunks.append(chunk)

    # ── Build sub-chunk dicts ─────────────────────────────────────────────────
    sub_chunks = []
    for chunk_text in filtered_chunks:
        sub_chunks.append({
            "header"     : section["header"],
            "context"    : section["context"],
            "content"    : chunk_text,
            "token_count": count_tokens(chunk_text),
        })

    return sub_chunks


print("Semantic chunking functions defined.")
print(f"  Primary  : semchunk (token_counter = count_tokens)")
print(f"  Fallback : regex sentence splitter (no NLTK data required)")
print(f"  Max tokens per chunk : {TOKEN_THRESHOLD}")
print(f"  Min tokens per chunk : {MIN_CHUNK_TOKENS}")

Semantic chunking functions defined.
  Primary  : semchunk (token_counter = count_tokens)
  Fallback : regex sentence splitter (no NLTK data required)
  Max tokens per chunk : 100
  Min tokens per chunk : 20


---
## 8. Final Chunk Output

Assemble the final `chunks` list by routing each section through the appropriate path:

- Short sections → single chunk
- Long sections → `semantic_chunk()` → multiple sub-chunks

Each chunk receives a complete metadata dict conforming to the schema required for vector indexing:

```python
{
    "document"   : str,   # Source document name (no extension)
    "section"    : str,   # Header breadcrumb path
    "chunk_text" : str,   # The text content to embed
    "token_count": int,   # Token count (matches embedding model tokenizer)
    "chunk_id"   : int,   # Sequential identifier (0-indexed)
}
```

In [20]:
def build_chunks(
    doc_name: str,
    sections: list[dict[str, Any]],
    max_tokens: int = TOKEN_THRESHOLD,
    min_tokens: int = MIN_CHUNK_TOKENS,
) -> list[dict[str, Any]]:
    """
    Assemble the final chunk list from all document sections.

    Routes each section through pass-through or semantic chunking based
    on token count. Attaches full metadata to every output chunk.

    Args:
        doc_name : Document identifier (used in metadata, no file extension).
        sections : Annotated section list from analyze_sections().
        max_tokens: Max tokens per chunk — passed to semantic_chunk().
        min_tokens: Min tokens per chunk — fragments below this are dropped.

    Returns:
        List of chunk dicts ready for vector embedding.
    """
    chunks: list[dict[str, Any]] = []
    chunk_id = 0

    for section in sections:
        content = section["content"]

        # Skip empty sections (e.g., heading with no body text)
        if not content.strip():
            continue

        if section["needs_chunking"]:
            # Long section: apply semantic chunking
            sub_chunks = semantic_chunk(section, max_tokens=max_tokens, min_tokens=min_tokens)
            for sub in sub_chunks:
                chunks.append({
                    "document"   : doc_name,
                    "section"    : sub["context"],
                    "chunk_text" : sub["content"],
                    "token_count": sub["token_count"],
                    "chunk_id"   : chunk_id,
                })
                chunk_id += 1
        else:
            # Short section: pass through as a single chunk
            tc = section["token_count"]
            if tc < min_tokens:
                continue  # Drop fragments below minimum
            chunks.append({
                "document"   : doc_name,
                "section"    : section["context"],
                "chunk_text" : content,
                "token_count": tc,
                "chunk_id"   : chunk_id,
            })
            chunk_id += 1

    return chunks


# ── Build Chunks ──────────────────────────────────────────────────────────────

doc_name = Path(TARGET_DOC).stem  # "LIFTOFF_DOE_VPP_2023"

print(f"Building chunks for '{doc_name}'...")
chunks = build_chunks(doc_name, sections)

print(f"\nChunking complete.")
print(f"  Total chunks produced : {len(chunks)}")

chunk_tokens = [c["token_count"] for c in chunks]
print(f"  Min tokens per chunk  : {min(chunk_tokens)}")
print(f"  Max tokens per chunk  : {max(chunk_tokens)}")
print(f"  Mean tokens per chunk : {sum(chunk_tokens) / len(chunk_tokens):.1f}")
import statistics
print(f"  Median tokens         : {statistics.median(chunk_tokens):.0f}")

Building chunks for 'potential-benefits-high-power-high-capacity-batteries-january-2020'...

Chunking complete.
  Total chunks produced : 526
  Min tokens per chunk  : 20
  Max tokens per chunk  : 111
  Mean tokens per chunk : 70.2
  Median tokens         : 72


In [21]:
# ── Inspect Sample Chunks ─────────────────────────────────────────────────────

print("Sample chunks (first 3):")
print("=" * 70)
for chunk in chunks[:3]:
    print(json.dumps(chunk, indent=2, ensure_ascii=False))
    print("─" * 70)

Sample chunks (first 3):
{
  "document": "potential-benefits-high-power-high-capacity-batteries-january-2020",
  "section": "Executive Summary",
  "chunk_text": "The ability of the United States (U.S.) electric power system (i.e., the electric grid) to reliably meet customer demand is crucial to our economy and national security. The increasing adoption of variable generation technologies and dynamic changes in customer demand are creating the need for enhanced grid flexibility to ensure the continued reliability, resilience, and security of the electric power system.",
  "token_count": 72,
  "chunk_id": 0
}
──────────────────────────────────────────────────────────────────────
{
  "document": "potential-benefits-high-power-high-capacity-batteries-january-2020",
  "section": "Executive Summary",
  "chunk_text": "Batteries and other energy storage technologies that have the capability to both supply and absorb electrical power (bidirectional electrical energy storage) can provide flexib

In [22]:
# ── Final Chunk DataFrame ─────────────────────────────────────────────────────

df_chunks = pd.DataFrame(chunks)

# Add a preview column for quick scanning
df_chunks["preview"] = df_chunks["chunk_text"].str[:80].str.replace("\n", " ") + "..."

print(f"Final chunks ({len(df_chunks)} total):")
display_cols = ["chunk_id", "token_count", "section", "preview"]
print(df_chunks[display_cols].to_string(index=False))

Final chunks (526 total):
 chunk_id  token_count                                               section                                                                             preview
        0           72                                     Executive Summary The ability of the United States (U.S.) electric power system (i.e., the electri...
        1           38                                     Executive Summary Batteries and other energy storage technologies that have the capability to both...
        2           89                             Report Scope and Approach This report describes opportunities for high-power, high-capacity batteries to i...
        3           69                             Report Scope and Approach Dimensional requirements for storage technologies vary by duration (intra-hour, ...
        4           73                             Report Scope and Approach The report also describes efforts by the U.S. Department of Energy (DOE), indust...
        

In [23]:
# ── Token Distribution Summary ────────────────────────────────────────────────

print("Token count distribution across final chunks:")
print(df_chunks["token_count"].describe().round(1).to_string())

# Verify schema completeness
required_keys = {"document", "section", "chunk_text", "token_count", "chunk_id"}
schema_ok = all(required_keys.issubset(c.keys()) for c in chunks)
print(f"\nSchema validation : {'PASS' if schema_ok else 'FAIL'}")
print(f"chunks variable   : list of {len(chunks)} dicts")
print("\nPipeline complete. 'chunks' is ready for embedding.")

Token count distribution across final chunks:
count    526.0
mean      70.2
std       20.1
min       20.0
25%       56.0
50%       72.0
75%       87.0
max      111.0

Schema validation : PASS
chunks variable   : list of 526 dicts

Pipeline complete. 'chunks' is ready for embedding.


---
## 9. Save Chunks to Disk

Persist the final chunks as a JSON file for downstream use (embedding pipelines, vector indexing, auditing).

In [24]:
OUTPUT_DIR = Path("chunks")
OUTPUT_DIR.mkdir(exist_ok=True)

output_path = OUTPUT_DIR / f"{doc_name}_chunks.json"

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(chunks, f, indent=2, ensure_ascii=False)

print(f"Saved {len(chunks)} chunks → {output_path}")

Saved 526 chunks → chunks/potential-benefits-high-power-high-capacity-batteries-january-2020_chunks.json
